# Evaluate the fine-tuned Ettin-32M model

Run this after full HPC training. Export `final_model/` and `run.json` as described in `hpc-unibz.md`, then upload the archive to Google Drive. The pretrained `jhu-clsp/ettin-encoder-32m` checkpoint is not your fine-tuned classifier.

Choose a GPU runtime. This notebook loads the saved two-output classifier with its tokenizer and fixed threshold, and evaluates the same frozen public test revision. Test results are for reporting, not hyperparameter selection. Timing and optional profiler FLOPs are local measurements, not official competition scores.

The Colab evaluation dependencies differ from the pinned HPC training environment to coexist with current Colab packages. Installed versions are recorded with results. After an install changes packages, restart the session before loading the model.


In [ ]:
# Evaluation dependencies compatible with current Colab packages.
# Keep Colab's GPU-enabled PyTorch; the HPC training requirements stay unchanged.
import importlib.metadata as metadata
import subprocess
import sys
from packaging.requirements import Requirement

packages = [
    'transformers==5.23.0',
    'datasets==5.2.0',
    'huggingface-hub>=1.32.0,<3.0',
]
try:
    gcsfs_dependencies = metadata.requires('gcsfs') or []
except metadata.PackageNotFoundError:
    gcsfs_dependencies = []
for dependency in gcsfs_dependencies:
    requirement = Requirement(dependency)
    if requirement.name == 'fsspec' and (requirement.marker is None or requirement.marker.evaluate()):
        packages.append(str(requirement))
        break
print('Installing:', packages)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('If packages changed, restart the Colab session before continuing.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Change this path to the archive uploaded from the HPC.
ARCHIVE = Path('/content/drive/MyDrive/NLBSE-2027/ettin-32m-001.tar.gz')
WORK_DIR = Path('/content/ettin-32m-export')
RESULT_DIR = Path('/content/ettin-32m-results')
BATCH_SIZE = 16
MAX_TEST_SAMPLES = None  # Set to 16 for a quick check; None evaluates the full public test.
REPETITIONS = 1         # Set to 10 for a separate repeated forward benchmark.
PROFILE_FLOPS = False   # Supported-operation estimates; fused attention can be omitted.


In [ ]:
import json
import tarfile
import time
import contextlib
import torch
import transformers
import datasets
from packaging.version import Version
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

assert Version(torch.__version__.split('+')[0]) >= Version('2.6.0'), 'Use PyTorch >= 2.6'
assert ARCHIVE.is_file(), f'Upload the HPC export to {ARCHIVE}'
assert BATCH_SIZE > 0 and REPETITIONS > 0
assert MAX_TEST_SAMPLES is None or MAX_TEST_SAMPLES > 0
WORK_DIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(ARCHIVE, 'r:gz') as archive:
    archive.extractall(WORK_DIR, filter='data')
MODEL_DIR = WORK_DIR / 'final_model'
run = json.loads((WORK_DIR / 'run.json').read_text())
preprocessing = json.loads((MODEL_DIR / 'preprocessing.json').read_text())
LABELS = ['Vulnerability', 'MAT']
assert preprocessing['labels'] == LABELS
assert 0 < preprocessing['max_length'] <= 512
assert preprocessing['threshold'] == 0.5
print('Training model:', run['config']['model_name'])
print('Training instances:', run['train_instances'], 'step limit:', run['max_steps'])
if run['max_steps'] != -1 or run['max_train_samples'] is not None:
    print('WARNING: this export came from a training run with step/sample limits.')


In [ ]:
config = AutoConfig.from_pretrained(MODEL_DIR, local_files_only=True)
options = {}
if config.model_type == 'modernbert':
    # Transformers does not persist reference_compile in config.json.
    config.reference_compile = False
    options['attn_implementation'] = 'sdpa'
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, config=config, local_files_only=True, **options)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR, use_fast=preprocessing.get('use_fast_tokenizer', False), local_files_only=True)
assert model.config.num_labels == 2
assert [model.config.id2label[i] for i in range(2)] == LABELS
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()
print('Device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))

def tokenize(functions):
    return tokenizer(functions, return_tensors='pt', padding='max_length', truncation=True,
                     max_length=preprocessing['max_length']).to(device)

with torch.inference_mode():
    example_probs = torch.sigmoid(model(**tokenize(['int f() { /* TODO: fix */ return 0; }'])).logits)
print('Example probabilities [Vulnerability, MAT]:', example_probs.cpu().tolist())
print('Example predictions:', (example_probs > preprocessing['threshold']).int().cpu().tolist())


In [ ]:
test = load_dataset(run['prepared']['dataset_name'],
                    revision=run['prepared']['dataset_revision'], split='test')
assert len(test) == run['prepared']['split_sizes']['test'], 'Unexpected public test size'
if MAX_TEST_SAMPLES is not None:
    test = test.select(range(min(MAX_TEST_SAMPLES, len(test))))
print('Evaluating', len(test), 'functions')


In [ ]:
def validate_labels(labels):
    if len(labels) != 2 or any(value not in (0, 1) for value in labels):
        raise ValueError(f"Expected binary [weakness, MAT] labels, got {labels!r}")
    return [float(value) for value in labels]


In [ ]:
def classification_metrics(labels, predictions):
    if not labels or len(labels) != len(predictions):
        raise ValueError("Need equally sized, nonempty labels and predictions")
    for row in labels + predictions:
        validate_labels(row)
    result = {}
    for column, name in enumerate(LABELS):
        tp = sum(y[column] == 1 and p[column] == 1 for y, p in zip(labels, predictions))
        fp = sum(y[column] == 0 and p[column] == 1 for y, p in zip(labels, predictions))
        tn = sum(y[column] == 0 and p[column] == 0 for y, p in zip(labels, predictions))
        fn = sum(y[column] == 1 and p[column] == 0 for y, p in zip(labels, predictions))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        result[name] = dict(accuracy=(tp + tn) / len(labels), precision=precision,
                            recall=recall, f1=2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0,
                            tp=tp, fp=fp, tn=tn, fn=fn)
    result["mean_f1"] = sum(result[name]["f1"] for name in LABELS) / 2
    return result


In [ ]:
labels, predictions, probabilities = [], [], []
forward_seconds, total_flops = 0.0, 0

def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize()

started = time.perf_counter()
with torch.inference_mode():
    for start in tqdm(range(0, len(test), BATCH_SIZE)):
        batch = test[start:start + BATCH_SIZE]
        labels.extend(batch['Label'])
        inputs = tokenize(batch['Function'])
        synchronize()
        begin = time.perf_counter()
        profile = torch.profiler.profile(with_flops=True) if PROFILE_FLOPS else contextlib.nullcontext()
        with profile as profiler:
            for _ in range(REPETITIONS):
                logits = model(**inputs).logits
        synchronize()
        forward_seconds += time.perf_counter() - begin
        if PROFILE_FLOPS:
            total_flops += sum(event.flops or 0 for event in profiler.key_averages())
        probs = torch.sigmoid(logits)
        probabilities.extend(probs.cpu().tolist())
        predictions.extend((probs > preprocessing['threshold']).int().cpu().tolist())

metrics = classification_metrics(labels, predictions)
metrics.update({
    'instances': len(test),
    'full_public_test': len(test) == run['prepared']['split_sizes']['test'],
    'forward_seconds_per_pass': forward_seconds / REPETITIONS,
    'gflops_per_pass': total_flops / REPETITIONS / 1e9 if PROFILE_FLOPS else None,
    'wall_seconds': time.perf_counter() - started,
    'repetitions': REPETITIONS, 'profiling_enabled': PROFILE_FLOPS,
    'eval_batch_size': BATCH_SIZE, 'precision': 'fp32', 'threshold': preprocessing['threshold'],
    'label_order': LABELS, 'training_run': run,
    'parameters': sum(parameter.numel() for parameter in model.parameters()),
    'device': torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu',
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__, 'datasets_version': datasets.__version__,
})
RESULT_DIR.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
with (RESULT_DIR / 'predictions.jsonl').open('w') as handle:
    for index, (label, prediction, probs) in enumerate(zip(labels, predictions, probabilities)):
        handle.write(json.dumps(dict(index=index, labels=label, predictions=prediction, probabilities=probs)) + '\n')
print(json.dumps({key: metrics[key] for key in ['Vulnerability', 'MAT', 'mean_f1', 'instances', 'full_public_test', 'forward_seconds_per_pass', 'gflops_per_pass']}, indent=2))


FLOPs are estimates for operations supported by the PyTorch profiler and may omit fused SDPA attention. Timing includes profiler overhead when enabled and excludes tokenization/transfers. Compare models on the same hardware, precision, and batch size; these values do not establish electricity consumption. Use the organisers' evaluator in their common environment for the official efficiency score.


## Combined score
Same formula and limits as the original CodeBERT notebook. Enable `PROFILE_FLOPS = True` before evaluation to supply GFLOPs; use `REPETITIONS = 10` for the original repetition count. This is a local estimate: hardware, batch size, precision, and profiler coverage affect comparisons.


In [ ]:
# Same combined-score formula as CodeBERT_baseline.ipynb.
max_avg_runtime = 1000
max_avg_flops = 3_000_000

def score(avg_f1, avg_runtime, avg_flops):
    return (
        0.6 * avg_f1
        + 0.2 * max((max_avg_runtime - avg_runtime) / max_avg_runtime, 0)
        + 0.2 * max((max_avg_flops - avg_flops) / max_avg_flops, 0)
    )

avg_f1 = metrics['mean_f1']
avg_p = sum(metrics[name]['precision'] for name in LABELS) / len(LABELS)
avg_r = sum(metrics[name]['recall'] for name in LABELS) / len(LABELS)
# These measurements are already averaged across repetitions.
avg_runtime = metrics['forward_seconds_per_pass']
avg_flops = metrics['gflops_per_pass']

print('Average precision:', avg_p)
print('Average recall:', avg_r)
print('Average F1:', avg_f1)
print('Average runtime:', avg_runtime)
print('Average GFLOPs:', avg_flops)

total_score = None
if avg_flops is None:
    print('Score unavailable: set PROFILE_FLOPS = True and rerun evaluation, then this cell.')
else:
    total_score = round(score(avg_f1, avg_runtime, avg_flops), 4)
    print('Total score (local estimate):', total_score)
    if not metrics['full_public_test']:
        print('This score uses a subset of the public test set.')

metrics['competition_score_local'] = total_score
(RESULT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
total_score


In [ ]:
from google.colab import files
files.download(str(RESULT_DIR / 'metrics.json'))
# Optional: files.download(str(RESULT_DIR / 'predictions.jsonl'))
